# Ziqra.ai — Remote GPU Training (Kaggle Notebooks)

This notebook is **infrastructure only**. All training logic — model loading, LoRA, the dataset pipeline, the terminal UI — lives in `training/trainer/` in the project repo. This notebook installs dependencies, fetches the project onto the Kaggle GPU, and runs the exact same command you'd run locally or on Colab:

```
python -m backend.knowledge_distillation.training.trainer.train --config backend/knowledge_distillation/training/configs/<coach>.yaml
```

**Before running:**
- `backend/knowledge_distillation/` (the trainer, configs, prompts, and your prepared dataset) must be committed and pushed to your GitHub repo. Gemma additionally requires a Hugging Face account with its gated license accepted (Cell 2) — Qwen does not.
- In the notebook editor's right sidebar, turn **Internet** on (Settings → Internet → On) — `pip install` and `git clone` both need it, and it's off by default on Kaggle. Turning it on also disables submitting this notebook to code competitions that forbid internet access; irrelevant here.
- Under **Settings → Accelerator**, select **GPU T4 x2** (Kaggle's free-tier dual-T4 option) or whichever GPU accelerator your account has available.

This is a Kaggle port of `train_on_colab.ipynb` — same training logic, hyperparameters, and cell structure; only the platform plumbing (secrets, storage paths, GPU count, file download) differs. Kaggle-specific changes are called out in comments below.

## Cell 1 — Install dependencies

In [ ]:
# Kaggle's GPU image already ships torch with CUDA support and recent
# versions of most of these -- this install is here to (a) pin/ensure the
# packages exist regardless of which base image Kaggle is currently running,
# and (b) apply the same torchao floor Colab needed: PEFT's LoRA dispatcher
# hard-rejects (raises ImportError, not a graceful skip) any torchao below
# 0.16.0, and older Kaggle images can ship one below that floor.
#
# Requires Internet ON for this notebook (Settings -> Internet, right sidebar)
# -- off by default on Kaggle, unlike Colab where it's always on.
!pip install -q transformers peft accelerate bitsandbytes rich pyyaml "torchao>=0.16.0"


## Cell 2 — Authenticate to Hugging Face (required for gated models like Gemma)

In [ ]:
# Kaggle-specific: Colab reads secrets via google.colab.userdata; Kaggle's
# equivalent is kaggle_secrets.UserSecretsClient. Add the secret first via
# this notebook's Add-ons menu -> Secrets -> Add a new secret, name it
# HF_TOKEN, and toggle it on for this notebook (same idea as Colab's
# per-notebook secret access toggle).
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN"))

# Alternative if you don't want to use Kaggle Secrets -- pops up an
# interactive widget to paste your token into instead:
# from huggingface_hub import notebook_login
# notebook_login()


## Cell 3 — Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/tahapathan2603/Ziqra-ai-integration.git"

# Kaggle-specific: Colab's scratch disk is /content; Kaggle's is
# /kaggle/working. Unlike /content, /kaggle/working IS the notebook's
# persistent output -- see Cell 7/8 for why that removes the need for a
# separate "mount durable storage" step.
PROJECT_DIR = "/kaggle/working/Ziqra-ai-integration"

# Pull on every run, not just clone-if-missing -- otherwise a session that
# already has PROJECT_DIR from an earlier run silently keeps using a stale
# checkout and never picks up new commits (e.g. trainer fixes).
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull origin main

%cd {PROJECT_DIR}


## Cell 4 — Kaggle input dataset (optional)

In [ ]:
# Kaggle-specific: there is no Google Drive here. If your prepared dataset
# isn't committed to git, attach it as a Kaggle Dataset instead (Add Input,
# right sidebar) -- Kaggle mounts it read-only at /kaggle/input/<dataset-slug>/
# automatically, no mount step needed (unlike Colab's drive.mount()). Skip
# this cell if your dataset is already in the cloned repo.
#
# List whatever got attached so you can find the exact slug to copy from:
!ls /kaggle/input/ 2>/dev/null || echo "No Kaggle input datasets attached."

# Optional: uncomment and set the slug to match what's listed above.
# import shutil
# shutil.copytree(
#     "/kaggle/input/<your-dataset-slug>",
#     "backend/knowledge_distillation/training/data/datasets",
#     dirs_exist_ok=True,
# )


## Cell 5 — Configure paths and verify GPU

In [ ]:
# The only environment-specific "configuration" this notebook does: which
# coach's YAML to train, and confirming the GPU(s) are attached and visible
# to torch. Nothing about the training run itself is configured here --
# that all lives in training/configs/*.yaml, read entirely by the trainer.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Kaggle-specific: Kaggle's free-tier GPU accelerator is two T4s (T4 x2),
# vs. Colab's single GPU -- confirm torch actually sees both before training,
# since a misconfigured accelerator setting (e.g. "GPU T4 x1" or "None"
# selected under Settings -> Accelerator) would otherwise fail silently here
# and only surface later as an OOM or a CPU-speed training run.
import torch

device_count = torch.cuda.device_count()
print(f"torch sees {device_count} CUDA device(s):")
for i in range(device_count):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")
if device_count == 0:
    print("WARNING: no GPU visible to torch -- check Settings -> Accelerator.")
elif device_count == 1:
    print("Single GPU detected -- Cell 6 will run single-process, same as Colab.")
else:
    print(f"{device_count} GPUs detected -- see Cell 6 for the multi-GPU launch option.")

CONFIG_PATH = "backend/knowledge_distillation/training/configs/articulation.yaml"
# CONFIG_PATH = "backend/knowledge_distillation/training/configs/delivery.yaml"

print("Training with config:", CONFIG_PATH)


## Cell 6 — Run the trainer

In [ ]:
# This is the entire training step -- identical command whether it runs on
# Colab, Kaggle, RunPod, Lambda Labs, or your own machine. The Rich terminal
# UI (progress bar, live loss/LR/GPU-memory panel) renders directly in this
# cell's output.
#
# train.py uses relative imports (it is a package module, not a standalone
# script), and its own path is backend/knowledge_distillation/training/trainer/
# -- both facts mean it must be run with -m from the project root, not as a
# plain script path.
#
# Default below is single-process. On Kaggle's T4x2 this now trains on GPU 0
# only, safely -- train.py's _prevent_accidental_data_parallel() pins
# CUDA_VISIBLE_DEVICES=0 for exactly this case (single process, >1 GPU
# visible, quantization off). Confirmed necessary on a real Kaggle run: left
# unguarded, transformers.Trainer silently wraps the model in the legacy
# torch.nn.DataParallel whenever more than one GPU is visible to a single
# process, which crashed training on the first step (DataParallel's
# per-replica input splitting doesn't satisfy Gemma3's forward requirements).
# Nothing to configure here -- it's automatic, and a no-op on a single-GPU
# host like Colab.
!python -m backend.knowledge_distillation.training.trainer.train --config {CONFIG_PATH}

# Kaggle-specific opt-in: if Cell 5 reported 2 GPUs (T4 x2), you can use both
# via Hugging Face Accelerate (DistributedDataParallel under the hood --
# Trainer detects the multi-process launch and wraps the model itself, see
# training/configs/README.md's "Multi-GPU (e.g. Kaggle T4x2)" section) by
# commenting out the line above and uncommenting this one instead. The YAML
# config does not change either way -- only how this cell invokes it does.
# Under this launch mode, WORLD_SIZE>1 makes train.py skip the single-GPU
# pin above and leave both GPUs visible, as intended.
# !accelerate launch --multi_gpu --num_processes=2 -m backend.knowledge_distillation.training.trainer.train --config {CONFIG_PATH}


## Cell 7 — Save checkpoints

In [ ]:
# Kaggle-specific: Colab's /content is wiped on disconnect, so Cell 7 there
# copies checkpoints to Drive for durability. Kaggle has no such step to
# begin with -- PROJECT_DIR already lives under /kaggle/working (Cell 3),
# and /kaggle/working IS this notebook's persistent output: anything left
# there is saved automatically when you save a version of the notebook (and
# downloadable afterward from the notebook's Output tab), no extra mount or
# copy needed. This cell just confirms the checkpoints exist.
from pathlib import Path

import yaml

config = yaml.safe_load(open(CONFIG_PATH))
output_dir = Path(config["checkpointing"]["output_dir"])
checkpoints_dir = output_dir / "checkpoints"

if not checkpoints_dir.exists():
    print(f"No checkpoints found at {checkpoints_dir}")
else:
    print(f"Checkpoints present at {checkpoints_dir} (already under /kaggle/working -- ")
    print("saved automatically when you save a version of this notebook).")


## Cell 8 — Export the final LoRA adapter

In [ ]:
# The one artifact you actually need afterward: the small, final adapter,
# stored separately from the larger intermediate checkpoints (see Cell 7).
#
# Kaggle-specific: google.colab.files.download() (which triggers a browser
# download prompt) has no Kaggle equivalent -- Kaggle instead makes anything
# under /kaggle/working downloadable from the notebook's Output tab after you
# save a version. IPython.display.FileLink below gives you a clickable link
# to the same file without leaving the notebook, as an in-session shortcut.
import shutil
from pathlib import Path

import yaml
from IPython.display import FileLink, display

config = yaml.safe_load(open(CONFIG_PATH))
output_dir = Path(config["checkpointing"]["output_dir"])
adapters_dir = output_dir / "adapters"

if not adapters_dir.exists():
    raise FileNotFoundError(f"No adapter found at {adapters_dir} -- did training in Cell 6 complete?")

zip_path = shutil.make_archive(f"/kaggle/working/{output_dir.name}_adapter", "zip", adapters_dir)
print(f"Adapter zipped: {zip_path}")

display(FileLink(zip_path))
print("Click the link above to download now, or save a version of this notebook")
print("and download the same zip from its Output tab afterward.")
